In [68]:
import pandas as pd
from datetime import datetime


In [69]:
raw_data_path = '../data/raw'
processed_data_path = '../data/processed'

# Leagues:
# Premier-League - 9
# La-Liga - 12
# Serie-A - 11
# Ligue-1
# Bundesliga - 20

season = '2024-2025'
league = 'Premier-League'

# Load files
match_stats = pd.read_csv(f"{raw_data_path}/ags_match_stats_{league}_{season}.csv")
player_stats = pd.read_csv(f"{raw_data_path}/ags_player_stats_{league}_{season}.csv")
ags_odds = pd.read_csv(f"{raw_data_path}/historical_ags_odds_output_{league}_{season}.csv")

In [70]:
ags_odds[(ags_odds['player'].str.contains('Cunha'))&(ags_odds['match_date'].astype(str).str.contains('2025-05-10'))][['match_date','home_team','away_team']]

,match_date,home_team,away_team
37990,2025-05-10,Wolverhampton Wanderers,Brighton and Hove Albion
38006,2025-05-10,Wolverhampton Wanderers,Brighton and Hove Albion
38021,2025-05-10,Wolverhampton Wanderers,Brighton and Hove Albion
38058,2025-05-10,Wolverhampton Wanderers,Brighton and Hove Albion
38075,2025-05-10,Wolverhampton Wanderers,Brighton and Hove Albion


In [71]:
# 🗺️ Full Team Name Mapping: sot_odds ➝ player_stats
team_name_mapping = {
    'Premier-League':{
        'Manchester United': 'Manchester Utd',
        'Newcastle United': 'Newcastle Utd',
        'Nottingham Forest': "Nott'ham Forest",
        'West Ham United': 'West Ham',
        'Brighton and Hove Albion': 'Brighton',
        'Tottenham Hotspur': 'Tottenham',
        'Wolverhampton Wanderers': 'Wolves',
        'Arsenal': 'Arsenal',
        'Aston Villa': 'Aston Villa',
        'Bournemouth': 'Bournemouth',
        'Brentford': 'Brentford',
        'Chelsea': 'Chelsea',
        'Crystal Palace': 'Crystal Palace',
        'Everton': 'Everton',
        'Fulham': 'Fulham',
        'Ipswich Town': 'Ipswich Town',
        'Leicester City': 'Leicester City',
        'Liverpool': 'Liverpool',
        'Manchester City': 'Manchester City',
        'Southampton': 'Southampton'},
    'La-Liga':{    
        "Alavés": "Alavés",
        "Athletic Bilbao": "Athletic Club",   # Mapped manually
        "Atlético Madrid": "Atlético Madrid",
        "Barcelona": "Barcelona",
        "CA Osasuna": "Osasuna",
        "Celta Vigo": "Celta Vigo",
        "Espanyol": "Espanyol",
        "Getafe": "Getafe",
        "Girona": "Girona",
        "Las Palmas": "Las Palmas",
        "Leganés": "Leganés",
        "Mallorca": "Mallorca",
        "Rayo Vallecano": "Rayo Vallecano",
        "Real Betis": "Betis",
        "Real Madrid": "Real Madrid",
        "Real Sociedad": "Real Sociedad",
        "Sevilla": "Sevilla",
        "Valencia": "Valencia",
        "Valladolid": "Valladolid",
        "Villarreal": "Villarreal"

        },
    'Serie-A':{
        "AC Milan": "AC Milan",
        "AS Roma": "Roma",
        "Atalanta BC": "Atalanta",
        "Bologna": "Bologna",
        "Cagliari": "Cagliari",
        "Empoli": "Empoli",
        "Fiorentina": "Fiorentina",
        "Genoa": "Genoa",
        "Hellas Verona": "Verona",
        "Inter Milan": "Inter",
        "Juventus": "Juventus",
        "Lazio": "Lazio",
        "Lecce": "Lecce",
        "Monza": "Monza",
        "Napoli": "Napoli",
        "Parma": "Parma",
        "Roma": "Roma",
        "Torino": "Torino",
        "Udinese": "Udinese",
        "Venezia": "Venezia",
        "Como": "Como"
}
}

team_name_mapping = team_name_mapping[league]

# # 🔁 Apply mapping to home and away teams in sot_odds
ags_odds["home_team"] = ags_odds["home_team"].replace(team_name_mapping)
ags_odds["away_team"] = ags_odds["away_team"].replace(team_name_mapping)




In [72]:
# Convert match_date in match_stats and player_stats to 'YYYY-MM-DD'
def standardize_date(date_str):
    return datetime.strptime(date_str, "%A %B %d, %Y").strftime("%Y-%m-%d")

player_stats["match_date"] = player_stats["match_date"].apply(standardize_date)
match_stats["match_date"] = match_stats["match_date"].apply(standardize_date)

# In sot_odds, convert ISO datetime string to just date
ags_odds["match_date"] = pd.to_datetime(ags_odds["match_date"]).dt.strftime("%Y-%m-%d")


In [73]:
# Combine Over/Under odds while retaining bookmaker
odds_pivot = ags_odds.pivot_table(
    index=["match_date", "player", "home_team", "away_team", "bookmaker"],
    columns="direction",
    values="odds",
    aggfunc="first"
).reset_index()

# Flatten column names
odds_pivot.columns.name = None
odds_pivot = odds_pivot.rename(columns={"Yes": "ags_odds"})


In [74]:
# 🟢 Merge for Home Team Players
home_merge = pd.merge(
    player_stats,
    odds_pivot,
    how="inner",
    left_on=["match_date", "player", "team"],
    right_on=["match_date", "player", "home_team"]
)

# 🟣 Merge for Away Team Players
away_merge = pd.merge(
    player_stats,
    odds_pivot,
    how="inner",
    left_on=["match_date", "player", "team"],
    right_on=["match_date", "player", "away_team"]
)

# 🧩 Combine both home and away player odds data
player_odds = pd.concat([home_merge, away_merge], ignore_index=True)


In [75]:
# Merge in team-level match stats for additional context
final_df = pd.merge(
    player_odds,
    match_stats,
    how="left",
    on=["match_date", "team"],
    suffixes=("", "_dup")  # prevent _x / _y confusion
)

# Drop duplicate columns (e.g., 'team_xG_dup', etc.)
dup_cols = [col for col in final_df.columns if col.endswith("_dup")]
final_df = final_df.drop(columns=dup_cols)


In [76]:
# Drop any unnamed index columns
final_df = final_df.drop(columns=[col for col in final_df.columns if "Unnamed" in col])



In [77]:
final_df[(final_df['player'].str.contains('Cunha'))&(final_df['match_date'].astype(str).str.contains('2025-05-10'))][['match_date','home_team','away_team']]

,match_date,home_team,away_team
14299,2025-05-10,Wolves,Brighton
14300,2025-05-10,Wolves,Brighton
14301,2025-05-10,Wolves,Brighton
14302,2025-05-10,Wolves,Brighton
14303,2025-05-10,Wolves,Brighton


In [78]:
# Clean and convert age column from "YY-DDD" to decimal years
def age_to_decimal(age_str):
    try:
        years, days = map(int, age_str.split("-"))
        return round(years + days / 365, 2)
    except:
        return None

final_df["age_decimal"] = final_df["age"].apply(age_to_decimal)

# Reorder columns so 'age_decimal' comes after 'age'
if "age" in final_df.columns and "age_decimal" in final_df.columns:
    cols = list(final_df.columns)
    age_idx = cols.index("age")
    cols.insert(age_idx + 1, cols.pop(cols.index("age_decimal")))
    final_df = final_df[cols]

# Optional: Drop the original 'age' column if no longer needed
final_df = final_df.drop(columns=["age"])

In [79]:
# Check for duplicate rows per player per match (for reference)
duplicates_check = (
    final_df.groupby(["match_date", "player", "team"])
    .agg(bookmaker_count=("bookmaker", "nunique"), total_rows=("bookmaker", "count"))
    .reset_index()
)

In [80]:
odds_pivot

,match_date,player,home_team,away_team,bookmaker,ags_odds
0,2024-08-16,Adama Traoré,Manchester Utd,Fulham,Bovada,6.00
1,2024-08-16,Alejandro Garnacho,Manchester Utd,Fulham,Bovada,2.95
2,2024-08-16,Alex Iwobi,Manchester Utd,Fulham,Bovada,5.75
3,2024-08-16,Amad Diallo,Manchester Utd,Fulham,Bovada,3.00
4,2024-08-16,Andreas Pereira,Manchester Utd,Fulham,Bovada,7.00
...,...,...,...,...,...,...
41213,2025-05-25,Yves Bissouma,Tottenham,Brighton,BetMGM,12.50
41214,2025-05-25,Yves Bissouma,Tottenham,Brighton,Bovada,12.00
41215,2025-05-25,Yves Bissouma,Tottenham,Brighton,DraftKings,14.00
41216,2025-05-25,Zach Abbott,Nott'ham Forest,Chelsea,Bovada,21.00


In [81]:
# Step 1: Get the best odds per player-match-line
best_odds = (
    final_df.sort_values("ags_odds", ascending=False)
    .drop_duplicates(subset=["match_date", "player", "team"], keep="first")
)


# Step 3: Combine and deduplicate again by full bookmaker record
final_df = best_odds.drop_duplicates(
    subset=["match_date", "player", "team", "bookmaker"],
    keep="first"
).reset_index(drop=True)


In [82]:
final_df[['player','ags_odds']]

,player,ags_odds
0,Luke Thomas,56.00
1,Adam Smith,56.00
2,Luke Thomas,51.00
3,Jean-Clair Todibo,51.00
4,Adam Smith,51.00
...,...,...
8184,Erling Haaland,1.56
8185,Mohamed Salah,1.54
8186,Erling Haaland,1.50
8187,Erling Haaland,1.45


In [83]:
# Save cleaned data if needed
final_df.to_csv(f'{processed_data_path}/ags_consolidated_{league}_{season}.csv', index=False)

# Preview result
print(final_df.shape)
final_df.head()

(8189, 45)


,match_date,player,team,position,age_decimal,minutes,goals,shots,shots_on_target,xg,...,team_possession,opp_possession,team_passing_accuracy,opp_passing_accuracy,opp_shots_on_target,opp_shots,team_saves,opp_saves,team_saves_faced,opp_saves_faced
0,2025-04-20,Luke Thomas,Leicester City,LB,23.86,90,0,0,0,0.0,...,42,58,76,83,8,28,7,0,8,0
1,2025-05-20,Adam Smith,Bournemouth,RB,34.06,22,0,0,0,0.0,...,43,57,83,89,5,12,2,1,5,2
2,2025-04-12,Luke Thomas,Leicester City,LB,23.84,60,0,1,0,0.1,...,41,59,78,85,5,19,5,4,5,6
3,2025-02-22,Jean-Clair Todibo,West Ham,CB,25.15,61,0,0,0,0.0,...,32,68,77,86,2,20,2,1,2,2
4,2024-11-02,Adam Smith,Bournemouth,RB,33.51,90,0,1,0,0.6,...,36,64,82,87,4,18,3,4,4,6
